# Advanced Topics: Sparse Transformers & Long Context Optimization

> **Goal**: Understand cutting-edge techniques for handling long sequences
>
> **Topics**:
> - Sparse Attention Patterns
> - Flash Attention
> - Ring Attention
> - Positional Interpolation
> - Sliding Window Attention
>
> **Prerequisites**: Module 07 (Transformers)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Problem: Standard Attention Complexity

### Standard Self-Attention

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

**Complexity**: $O(n^2 \cdot d)$ where $n$ is sequence length

**Problem**: Quadratic memory and compute with sequence length

| Sequence Length | Memory (GB @ FP16) |
|-----------------|--------------------|
| 1K | 0.01 |
| 4K | 0.13 |
| 16K | 2.0 |
| 64K | 32.0 |
| 128K | 128.0 |

**Solutions**:
1. **Sparse Attention**: Only attend to subset of tokens
2. **Flash Attention**: Fused kernel, tiling
3. **Linear Attention**: Approximate attention in O(n)
4. **Local + Global**: Sliding window + global tokens

## Part 1: Sparse Attention Patterns

### Pattern 1: Local (Sliding Window) Attention

Each token only attends to $w$ tokens before and after it.

**Complexity**: $O(n \cdot w \cdot d)$ where $w \ll n$

In [ ]:
class SlidingWindowAttention(nn.Module):
    """Local attention with sliding window.
    
    Used in: Longformer, BigBird
    """
    
    def __init__(self, d_model, n_heads, window_size=256, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.window_size = window_size
        
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)
        
        self.dropout = nn.Dropout(dropout)
    
    def create_sliding_window_mask(self, seq_len, device):
        """Create mask for sliding window attention.
        
        Returns:
            mask: (seq_len, seq_len) where mask[i,j]=1 if |i-j| <= window_size
        """
        mask = torch.zeros(seq_len, seq_len, device=device)
        
        for i in range(seq_len):
            start = max(0, i - self.window_size // 2)
            end = min(seq_len, i + self.window_size // 2 + 1)
            mask[i, start:end] = 1
        
        return mask
    
    def forward(self, x):
        """
        Args:
            x: (batch, seq_len, d_model)
        """
        batch_size, seq_len, _ = x.shape
        
        # Project to Q, K, V
        Q = self.W_Q(x).view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_K(x).view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_V(x).view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        
        # Compute attention scores
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        # Apply sliding window mask
        mask = self.create_sliding_window_mask(seq_len, x.device)
        scores = scores.masked_fill(mask.unsqueeze(0).unsqueeze(0) == 0, float('-inf'))
        
        # Softmax and apply to values
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        
        output = torch.matmul(attn_weights, V)
        output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        output = self.W_O(output)
        
        return output, attn_weights

# Visualize sliding window pattern
def visualize_attention_pattern(mask, title="Attention Pattern"):
    plt.figure(figsize=(8, 8))
    sns.heatmap(mask.cpu().numpy(), cmap='Blues', cbar=False, square=True)
    plt.title(title)
    plt.xlabel('Key Position')
    plt.ylabel('Query Position')
    plt.show()

# Test
seq_len = 64
window_attn = SlidingWindowAttention(d_model=512, n_heads=8, window_size=16)
mask = window_attn.create_sliding_window_mask(seq_len, torch.device('cpu'))
visualize_attention_pattern(mask, "Sliding Window Attention (window=16)")

print(f"Sparsity: {(mask == 0).float().mean():.2%} of attention matrix is zero")

### Pattern 2: Strided (Dilated) Attention

Attend to every $k$-th token. Captures long-range dependencies with fixed compute.

**Used in**: Longformer, Reformer

In [ ]:
def create_strided_mask(seq_len, stride=4):
    """Create mask for strided attention.
    
    Token i attends to tokens i, i±stride, i±2*stride, ...
    """
    mask = torch.zeros(seq_len, seq_len)
    
    for i in range(seq_len):
        for j in range(0, seq_len, stride):
            mask[i, j] = 1
    
    return mask

# Visualize
strided_mask = create_strided_mask(seq_len=64, stride=4)
visualize_attention_pattern(strided_mask, "Strided Attention (stride=4)")
print(f"Sparsity: {(strided_mask == 0).float().mean():.2%}")

### Pattern 3: Global + Local Attention

- **Local**: Sliding window for all tokens
- **Global**: Some tokens attend to everything

**Used in**: BigBird, Longformer

**Example**: [CLS] token in BERT is global

In [ ]:
def create_global_local_mask(seq_len, window_size=8, global_tokens=4):
    """Combine local sliding window with global tokens.
    
    Args:
        seq_len: Sequence length
        window_size: Size of local window
        global_tokens: Number of initial tokens that attend globally
    """
    mask = torch.zeros(seq_len, seq_len)
    
    # Global tokens attend to everything and everything attends to them
    mask[:global_tokens, :] = 1
    mask[:, :global_tokens] = 1
    
    # Local sliding window for remaining tokens
    for i in range(global_tokens, seq_len):
        start = max(global_tokens, i - window_size // 2)
        end = min(seq_len, i + window_size // 2 + 1)
        mask[i, start:end] = 1
    
    return mask

# Visualize
global_local_mask = create_global_local_mask(seq_len=64, window_size=8, global_tokens=4)
visualize_attention_pattern(global_local_mask, "Global + Local Attention")
print(f"Sparsity: {(global_local_mask == 0).float().mean():.2%}")

## Part 2: Flash Attention

### The Problem with Standard Attention

Standard attention implementation:
```python
S = Q @ K.T  # (n, n) - materialize full attention matrix
P = softmax(S)
O = P @ V
```

**Issue**: Materializing $(n \times n)$ attention matrix requires $O(n^2)$ memory.

### Flash Attention Solution

**Key insight**: Never materialize full attention matrix. Use **tiling** and **recomputation**.

**Algorithm** (simplified):
1. Split $Q, K, V$ into blocks
2. Compute attention blockwise
3. Fuse operations in single kernel
4. Recompute in backward pass (trade compute for memory)

**Benefits**:
- 2-4x faster
- 10-20x less memory
- No approximation (exact attention)

### Conceptual Implementation

In [ ]:
class FlashAttentionConcept(nn.Module):
    """Conceptual Flash Attention (simplified for understanding).
    
    In practice, use: from flash_attn import flash_attn_func
    """
    
    def __init__(self, block_size=64):
        super().__init__()
        self.block_size = block_size
    
    def forward(self, Q, K, V):
        """
        Block-wise attention computation.
        
        Args:
            Q, K, V: (batch, n_heads, seq_len, d_k)
        """
        batch_size, n_heads, seq_len, d_k = Q.shape
        block_size = self.block_size
        
        # Initialize output
        O = torch.zeros_like(V)
        
        # Online softmax statistics
        l = torch.zeros(batch_size, n_heads, seq_len, 1, device=Q.device)
        m = torch.full((batch_size, n_heads, seq_len, 1), float('-inf'), device=Q.device)
        
        # Process in blocks (simplified - real implementation is more complex)
        num_blocks = (seq_len + block_size - 1) // block_size
        
        for i in range(num_blocks):
            # Query block
            q_start = i * block_size
            q_end = min((i + 1) * block_size, seq_len)
            Q_block = Q[:, :, q_start:q_end, :]
            
            for j in range(num_blocks):
                # Key-Value block
                k_start = j * block_size
                k_end = min((j + 1) * block_size, seq_len)
                K_block = K[:, :, k_start:k_end, :]
                V_block = V[:, :, k_start:k_end, :]
                
                # Compute attention for this block
                S_block = torch.matmul(Q_block, K_block.transpose(-2, -1)) / math.sqrt(d_k)
                
                # Update statistics (online softmax)
                m_block = S_block.max(dim=-1, keepdim=True)[0]
                m_prev = m[:, :, q_start:q_end, :]
                m_new = torch.maximum(m_prev, m_block)
                
                # Compute attention probabilities
                P_block = torch.exp(S_block - m_new)
                
                # Update output
                O[:, :, q_start:q_end, :] += torch.matmul(P_block, V_block)
                
                # Update statistics
                l[:, :, q_start:q_end, :] += P_block.sum(dim=-1, keepdim=True)
                m[:, :, q_start:q_end, :] = m_new
        
        # Normalize
        O = O / l
        
        return O

print("Flash Attention Key Ideas:")
print("1. Never materialize full attention matrix")
print("2. Tiling: Process in small blocks")
print("3. Fused kernel: All ops in one GPU kernel")
print("4. Online softmax: Incremental computation")
print("5. Recomputation: Trade compute for memory in backward pass")

## Part 3: Positional Encoding Extensions

### Problem: Fixed Position Embeddings

Standard sinusoidal encoding: Trained on sequences up to length $L$, fails on $L' > L$.

### Solution 1: Rotary Position Embeddings (RoPE)

**Used in**: LLaMA, GPT-NeoX, PaLM

**Idea**: Rotate query and key vectors by position-dependent angles.

In [ ]:
class RotaryPositionEmbedding(nn.Module):
    """Rotary Position Embedding (RoPE).
    
    Paper: RoFormer (Su et al., 2021)
    """
    
    def __init__(self, d_model, max_len=2048, base=10000):
        super().__init__()
        self.d_model = d_model
        
        # Compute frequencies
        inv_freq = 1.0 / (base ** (torch.arange(0, d_model, 2).float() / d_model))
        self.register_buffer('inv_freq', inv_freq)
    
    def forward(self, x, seq_len):
        """
        Args:
            x: (batch, seq_len, d_model)
        """
        # Generate position indices
        t = torch.arange(seq_len, device=x.device).type_as(self.inv_freq)
        
        # Compute angles
        freqs = torch.einsum('i,j->ij', t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        
        # Apply rotation
        cos = emb.cos()[None, :, None, :]
        sin = emb.sin()[None, :, None, :]
        
        # Rotate x
        x_rotated = self.rotate_half(x) * sin + x * cos
        
        return x_rotated
    
    def rotate_half(self, x):
        """Rotate half the dimensions."""
        x1, x2 = x[..., :x.shape[-1]//2], x[..., x.shape[-1]//2:]
        return torch.cat([-x2, x1], dim=-1)

# Test RoPE
rope = RotaryPositionEmbedding(d_model=512)
x = torch.randn(2, 100, 512)
x_rope = rope(x, seq_len=100)

print(f"Input shape: {x.shape}")
print(f"Output shape: {x_rope.shape}")
print("\nRoPE benefits:")
print("- Relative position encoding")
print("- Better extrapolation to longer sequences")
print("- Used in LLaMA, GPT-NeoX")

### Solution 2: ALiBi (Attention with Linear Biases)

**Paper**: Train Short, Test Long (Press et al., 2022)

**Idea**: Add linear bias to attention scores based on distance.

In [ ]:
class ALiBi(nn.Module):
    """Attention with Linear Biases.
    
    Instead of positional embeddings, directly bias attention scores.
    """
    
    def __init__(self, n_heads):
        super().__init__()
        self.n_heads = n_heads
        
        # Compute head-specific slopes
        slopes = torch.Tensor(self._get_slopes(n_heads))
        self.register_buffer('slopes', slopes)
    
    def _get_slopes(self, n_heads):
        """Geometric sequence of slopes."""
        def get_slopes_power_of_2(n):
            start = 2 ** (-(2 ** -(math.log2(n) - 3)))
            ratio = start
            return [start * (ratio ** i) for i in range(n)]
        
        if math.log2(n_heads).is_integer():
            return get_slopes_power_of_2(n_heads)
        else:
            closest_power_of_2 = 2 ** math.floor(math.log2(n_heads))
            return get_slopes_power_of_2(closest_power_of_2) + \
                   self._get_slopes(2 * closest_power_of_2)[::2][:n_heads - closest_power_of_2]
    
    def forward(self, seq_len, device):
        """Generate ALiBi bias matrix.
        
        Returns:
            bias: (1, n_heads, seq_len, seq_len)
        """
        # Distance matrix
        arange = torch.arange(seq_len, device=device)
        distance = arange.unsqueeze(0) - arange.unsqueeze(1)
        distance = distance.abs()
        
        # Apply slopes
        bias = -distance.unsqueeze(0) * self.slopes.view(-1, 1, 1)
        
        return bias.unsqueeze(0)

# Visualize ALiBi biases
alibi = ALiBi(n_heads=8)
bias = alibi.forward(seq_len=32, device='cpu')

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, ax in enumerate(axes.flat):
    sns.heatmap(bias[0, i].numpy(), ax=ax, cmap='coolwarm', center=0)
    ax.set_title(f'Head {i+1} (slope={alibi.slopes[i]:.4f})')
    ax.set_xlabel('Key Position')
    ax.set_ylabel('Query Position')

plt.suptitle('ALiBi: Attention Biases per Head')
plt.tight_layout()
plt.show()

print("\nALiBi advantages:")
print("- No position embeddings needed")
print("- Excellent extrapolation (train on 1K, test on 100K+)")
print("- Used in BLOOM, MPT")

## Summary: Choosing the Right Technique

| Technique | Complexity | Use Case | Models |
|-----------|------------|----------|--------|
| **Standard Attention** | $O(n^2)$ | n < 2K | BERT, GPT-2 |
| **Sliding Window** | $O(nw)$ | Local context | Longformer |
| **Global + Local** | $O(n(w+g))$ | Doc classification | BigBird |
| **Flash Attention** | $O(n^2)$ faster | Any, especially long | GPT-4, LLaMA |
| **RoPE** | Same | Relative positions | LLaMA, GPT-NeoX |
| **ALiBi** | Same | Extrapolation | BLOOM, MPT |
| **Linear Attention** | $O(n)$ | Very long (100K+) | Mamba, RWKV |

### Decision Tree

```
Sequence < 2K?
├─ Yes → Standard Attention + Flash Attention
└─ No → Is locality important?
    ├─ Yes → Sliding Window + Global tokens
    └─ No → Need exact attention?
        ├─ Yes → Flash Attention + better positional encoding (RoPE/ALiBi)
        └─ No → Linear Attention (Mamba, RWKV)
```

### Production Recommendations

**For most LLMs** (2024):
- **Attention**: Flash Attention 2
- **Positions**: RoPE or ALiBi
- **Long context**: Sliding window + RoPE interpolation

**Cutting edge** (research):
- State space models (Mamba)
- Hybrid architectures (attention + SSM)
- Memory-augmented transformers

## Further Reading

### Papers
1. **Longformer** (Beltagy et al., 2020)
2. **BigBird** (Zaheer et al., 2020)
3. **Flash Attention** (Dao et al., 2022)
4. **Flash Attention 2** (Dao, 2023)
5. **RoFormer (RoPE)** (Su et al., 2021)
6. **ALiBi** (Press et al., 2022)
7. **Mamba** (Gu & Dao, 2023)

### Implementations
- [Flash Attention GitHub](https://github.com/Dao-AILab/flash-attention)
- [Hugging Face Transformers](https://github.com/huggingface/transformers)
- [xFormers](https://github.com/facebookresearch/xformers)

This notebook covered the state-of-the-art in efficient transformers. Understanding these techniques is crucial for building production LLMs!